# Addison, TX — Land Value Tax Model

**Model type**: Split-rate 4:1 (land taxed at 4× the improvement rate)  
**Levy scope**: City of Addison only  
**Data source**: Dallas Central Appraisal District (DCAD) — Property/ParcelQuery MapServer, Layer 4  
**Tax year**: FY2025 rate ($0.6081/$100); validated against 2025 Certified EVR (DCAD, July 25 2025)  

## Column Mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `LNDVALUE` | DCAD certified market value for land |
| Improvement value | `IMPVALUE` | DCAD certified market value for improvements |
| Total market value | `CNTASSDVAL` | = LNDVALUE + IMPVALUE (pre-exemption) |
| Class code | `CLASSCD` | DCAD property class (1=SFR, 17=Commercial, etc.) |
| Parcel filter | Addison GIS portal parcel ID list | `PSTLCITY` is owner mailing address; centroid-in-polygon over-captures border parcels. City's own GIS portal is the authoritative jurisdiction list. |
| BPP exclusion | `CLASSCD NOT IN ('31','32','34','46','33')` | Excludes Business Personal Property |

**Assessment ratio**: 100% — Texas appraises at full market value  
**Millage source**: FY2025 ACFR confirmed rate — $0.6081 per $100 of assessed value  
**Homestead exemption**: 20% off appraised value for owner-occupied residential (city levy); approximated by applying to all CLASS 1/2/3/6 parcels since DCAD MapServer does not expose HS flags  
**Full exemptions**: Government (Town/City of Addison), educational (Greenhill School, Dallas ISD), religious (Whiterock Chapel) — identified via owner name matching

## 2025 Certified EVR Validation Baseline (DCAD, July 25 2025)

| | Parcels | Market Value | Taxable Value |
|---|---|---|---|
| Commercial | 726 | $5,247,358,990 | $4,642,146,959 |
| Business Personal Property | 2,724 | $986,170,970 | $960,212,773 |
| Residential | 2,375 | $1,324,787,400 | $1,018,191,223 |
| **Grand Total** | **5,825** | **$7,558,317,360** | **$6,620,550,955** |

Real property taxable (ex-BPP): **$5,660,338,182** → implied levy at $0.6081/$100 = **$34.4M**  

**Revenue gap note**: Model produces ~$38.4M (+11.6% vs certified $34.4M). Residual gap = unmodeled exemptions not exposed by MapServer: over-65/disabled ($60K each, per DCAD 2025 Tax Rates sheet), and nonprofit/government commercial properties not identifiable from owner names alone (~$280M estimated from EVR differential). Revenue-neutrality of the split-rate model is maintained within the modeled parcel set.

In [1]:
import sys
import json
import requests
from pathlib import Path
from shapely.geometry import Polygon

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'addison'
STATE_FIPS = '48'    # Texas
COUNTY_FIPS = '113'  # Dallas County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

MILLAGE_ADDISON = 0.608100  # FY2025 adopted rate (per $100 assessed value)

# 20% homestead exemption applied to CLASS 1/2/3/6 as proxy for owner-occupied residential;
# DCAD MapServer does not expose per-parcel HS flags
HOMESTEAD_RATE = 0.20

EXEMPT_OWNER_KEYWORDS = [
    'TOWN OF ADDISON', 'ADDISON TOWN OF', 'ADDISON CITY OF',
    'TOWN OF ADDSION',  # typo variant in data
    'GREENHILL SCHOOL',
    'WHITEROCK CHAPEL',
    'DALLAS ISD',       # school district facility at 3815 Spring Valley Rd ($84M)
]

BPP_CLASSES = ('31', '32', '33', '34', '46')
HOMESTEAD_CLASSES = ('1', '2', '3', '6')

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Fetch / Load Parcel Data

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
DCAD_QUERY_URL = 'https://maps.dcad.org/prdwa/rest/services/Property/ParcelQuery/MapServer/4/query'
TIGER_URL = 'https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_Current/MapServer/28/query'
ADDISON_GEOID = '4801240'
ADDISON_GIS_CSV = DATA_DIR / 'addison_gis_parcels.csv'


def _get_addison_boundary():
    params = {
        'where': f"GEOID='{ADDISON_GEOID}'",
        'outFields': 'NAME,GEOID,STATE',
        'returnGeometry': 'true',
        'outSR': '102100',
        'f': 'json',
    }
    resp = requests.get(TIGER_URL, params=params, timeout=30)
    resp.raise_for_status()
    features = resp.json().get('features', [])
    if not features:
        raise ValueError(f'GEOID {ADDISON_GEOID} not found in TIGERweb')
    attrs = features[0]['attributes']
    print(f"Boundary: {attrs['NAME']} (GEOID {attrs['GEOID']}, STATE {attrs['STATE']})")
    return features[0]['geometry']


def _download_dcad_addison():
    boundary_geom = _get_addison_boundary()

    # Authoritative Addison parcel ID list from the city's own GIS portal
    gis_ids = set(
        pd.read_csv(ADDISON_GIS_CSV, low_memory=False)['Parcel Identification Number']
        .astype(str).str.strip()
    )
    print(f'Addison GIS parcel list: {len(gis_ids):,} parcel IDs')

    bpp_list = "','".join(BPP_CLASSES)
    # POST required — boundary polygon JSON (~15KB) exceeds GET URL limits
    base_data = {
        'f': 'json',
        'where': f"CLASSCD NOT IN ('{bpp_list}')",
        'geometry': json.dumps(boundary_geom),
        'geometryType': 'esriGeometryPolygon',
        'spatialRel': 'esriSpatialRelIntersects',
        'inSR': '102100',
    }

    count = requests.post(DCAD_QUERY_URL, data={**base_data, 'returnCountOnly': 'true'}, timeout=30).json().get('count', 0)
    print(f'Parcels intersecting boundary (server count): {count:,}')

    all_features, offset = [], 0
    while True:
        resp = requests.post(DCAD_QUERY_URL, data={
            **base_data,
            'outFields': '*', 'returnGeometry': 'true', 'geometryPrecision': '6',
            'resultOffset': str(offset), 'resultRecordCount': '4000',
        }, timeout=60)
        resp.raise_for_status()
        resp_json = resp.json()
        features = resp_json.get('features', [])
        if not features:
            break
        for feat in features:
            attrs = feat['attributes']
            geom = feat.get('geometry')
            if geom and 'rings' in geom and geom['rings']:
                try:
                    attrs['geometry'] = Polygon(geom['rings'][0])
                except Exception:
                    attrs['geometry'] = None
            else:
                attrs['geometry'] = None
            all_features.append(attrs)
        offset += len(features)
        print(f'  Fetched {len(all_features)} records')
        if not resp_json.get('exceededTransferLimit', False):
            break

    result = gpd.GeoDataFrame(all_features, crs='EPSG:3857')
    result = result[result['geometry'].notna()].drop_duplicates(subset=['PARCELID']).copy()

    before = len(result)
    result = result[result['PARCELID'].astype(str).str.strip().isin(gis_ids)].copy()
    print(f'After GIS ID filter: {len(result):,} parcels (removed {before - len(result)} non-Addison)')

    return result.to_crs('EPSG:4326')


if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f'Loaded {len(gdf):,} parcels from cache')
else:
    gdf = _download_dcad_addison()
    gdf.to_parquet(PARCEL_PATH)
    print(f'Downloaded and cached {len(gdf):,} parcels')

print(f'\nParcel count: {len(gdf):,}')
print(gdf[['LNDVALUE', 'IMPVALUE', 'CNTASSDVAL']].describe())

Loaded 3,272 parcels from cache

Parcel count: 3,272
           LNDVALUE      IMPVALUE    CNTASSDVAL
count  3.272000e+03  3.272000e+03  3.272000e+03
mean   4.707233e+05  1.656844e+06  2.127568e+06
std    2.171421e+06  7.834118e+06  8.798624e+06
min    0.000000e+00  0.000000e+00  1.000000e+02
25%    8.000000e+04  3.305700e+05  4.330750e+05
50%    1.000000e+05  4.403350e+05  5.451300e+05
75%    1.440000e+05  5.514350e+05  7.112300e+05
max    7.617389e+07  1.393092e+08  1.458696e+08


## Section 3 — Classify and Validate

In [3]:
# Clip negative values
gdf['LNDVALUE'] = gdf['LNDVALUE'].fillna(0).clip(lower=0)
gdf['IMPVALUE'] = gdf['IMPVALUE'].fillna(0).clip(lower=0)
gdf['CNTASSDVAL'] = gdf['CNTASSDVAL'].fillna(0).clip(lower=0)

# Full exemption flag: government, school, religious owners
owner = gdf['OWNERNME1'].fillna('').str.upper()
gdf['full_exmp'] = owner.apply(
    lambda o: int(any(k in o for k in EXEMPT_OWNER_KEYWORDS))
)

# Homestead approximation: apply 20% city exemption to likely owner-occupied residential
gdf['hs_approx'] = gdf['CLASSCD'].isin(HOMESTEAD_CLASSES).astype(int)

# Property category mapping using DCAD class codes
def classify_parcel(row):
    cls = str(row.get('CLASSCD', '') or '').strip()
    if cls == '1':  return 'Single Family Residential'
    if cls == '2':  return 'Townhome / Rowhouse'
    if cls == '3':  return 'Condominium'
    if cls == '5':  return 'Large Multi-Family (5+ units)'
    if cls == '6':  return 'Small Multi-Family (2-4 units)'
    if cls in ('7', '8', '9', '10', '39'):  return 'Vacant Land'
    if cls == '11': return 'Vacant Land'   # Qualified Open Space — land-only or minimal improvement
    if cls == '17': return 'Commercial'
    if cls == '18': return 'Industrial'
    if cls == '24': return 'Vacant Land'   # Electric utility land — $0 improvement
    if cls == '35': return 'Other Residential'  # mobile home on leased land
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_parcel, axis=1)

# Override: $0 improvement → Vacant Land (regardless of class code)
gdf.loc[gdf['IMPVALUE'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f'\nFull-exempt parcels: {gdf["full_exmp"].sum()}')
print(f'Homestead-approx parcels: {gdf["hs_approx"].sum()}')
print(f'Total parcels: {len(gdf):,}')

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential         1388
Condominium                        583
Commercial                         488
Townhome / Rowhouse                427
Vacant Land                        261
Small Multi-Family (2-4 units)      84
Large Multi-Family (5+ units)       40
Industrial                           1
Name: count, dtype: int64

Full-exempt parcels: 100
Homestead-approx parcels: 2482
Total parcels: 3,272


## Section 4 — Current Tax Model

Texas levies at 100% market value. No fractional assessment ratio.

**Rate**: $0.6081 per $100 assessed value (FY2025 ACFR confirmed; same rate adopted for FY2026)  

**Addison exemptions (2025 DCAD Tax Rates sheet)**:

| Exemption | Amount | Modeled? |
|---|---|---|
| Optional homestead | 20% of appraised value | ✓ approximated via CLASS 1/2/3/6 |
| General homestead | $0 | N/A |
| Age 65 or older | $60,000 off appraised value | ✗ (flag not in MapServer) |
| Disabled person | $60,000 off appraised value | ✗ (flag not in MapServer) |
| Government / educational / religious | Full exemption | ✓ via owner name matching (Town of Addison, Dallas ISD, Greenhill School, Whiterock Chapel) |

**Validation against 2025 Certified EVR (DCAD, July 25 2025)**:  
- Certified real property taxable: $5,660.3M → implied levy $34.4M  
- This model taxable base: ~$6.3B → modeled levy ~$38.4M (~+11.6% gap)  
- Residual gap = unmodeled over-65/disabled exemptions + nonprofit/government commercial properties not identifiable by owner name (~$280M)  
- BPP ($960.2M taxable, ~$5.8M levy) excluded by design.

In [4]:
# Taxable values: apply homestead exemption to qualifying residential parcels
# Scale = 0.80 for homestead, 1.00 for all others
hs_scale = np.where(gdf['hs_approx'] == 1, 1.0 - HOMESTEAD_RATE, 1.0)
gdf['taxable_land_value'] = gdf['LNDVALUE'] * hs_scale
gdf['taxable_improvement_value'] = gdf['IMPVALUE'] * hs_scale

# Zero out fully-exempt parcels
gdf.loc[gdf['full_exmp'] == 1, 'taxable_land_value'] = 0.0
gdf.loc[gdf['full_exmp'] == 1, 'taxable_improvement_value'] = 0.0

# Current tax: city levy only
# MILLAGE_ADDISON is per $100; divide by 100 to get per-$ rate
gdf['current_tax'] = np.where(
    gdf['full_exmp'] == 1,
    0.0,
    (gdf['taxable_land_value'] + gdf['taxable_improvement_value']) * MILLAGE_ADDISON / 100.0
)

current_revenue = float(gdf['current_tax'].sum())
taxable_base = (gdf['taxable_land_value'] + gdf['taxable_improvement_value']).sum()

# 2025 Certified EVR (DCAD, July 25 2025) — same Jan 1 2025 values as MapServer data
# Commercial $4,642,146,959 + Residential $1,018,191,223
CERTIFIED_REAL_PROP_TAXABLE = 5_660_338_182
CERTIFIED_REAL_PROP_LEVY    = round(CERTIFIED_REAL_PROP_TAXABLE * MILLAGE_ADDISON / 100)  # $34.4M
CERTIFIED_BPP_TAXABLE       =   960_212_773
CERTIFIED_BPP_LEVY          = round(CERTIFIED_BPP_TAXABLE * MILLAGE_ADDISON / 100)         # $5.8M
CERTIFIED_TOTAL_TAXABLE     = 6_620_550_955  # real + BPP, after all exemptions

print(f'Modeled real-property revenue:         ${current_revenue:>15,.0f}')
print(f'Certified real-property levy (2025):   ${CERTIFIED_REAL_PROP_LEVY:>15,.0f}')
print(f'Gap vs. certified:                     {(current_revenue/CERTIFIED_REAL_PROP_LEVY - 1)*100:>+14.1f}%')
print()
print(f'Modeled taxable base:                  ${taxable_base/1e6:>14.1f}M')
print(f'Certified real property taxable:       ${CERTIFIED_REAL_PROP_TAXABLE/1e6:>14.1f}M')
print(f'  Commercial taxable:                  ${4_642_146_959/1e6:>14.1f}M')
print(f'  Residential taxable:                 ${1_018_191_223/1e6:>14.1f}M')
print()
print(f'Certified BPP (excluded by design):    ${CERTIFIED_BPP_LEVY:>15,.0f}')
print(f'Certified total taxable (real + BPP):  ${CERTIFIED_TOTAL_TAXABLE/1e6:>14.1f}M')

Modeled real-property revenue:         $     38,335,700
Certified real-property levy (2025):   $     34,420,516
Gap vs. certified:                              +11.4%

Modeled taxable base:                  $        6304.2M
Certified real property taxable:       $        5660.3M
  Commercial taxable:                  $        4642.1M
  Residential taxable:                 $        1018.2M

Certified BPP (excluded by design):    $      5,839,054
Certified total taxable (real + BPP):  $        6620.6M


## Section 5 — Split-Rate Model (4:1)

In [5]:
taxable = gdf[gdf['full_exmp'] == 0].copy()
print(f'Solving on {len(taxable):,} taxable parcels (excluded {gdf["full_exmp"].sum()} fully-exempt)')

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=float(taxable['current_tax'].sum()),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine with exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} per $1,000')
print(f'Improvement millage: {improvement_millage:.4f} per $1,000')
print(f'Land/improvement ratio: {land_millage/improvement_millage:.2f}:1')
print(f'Revenue check: ${new_revenue:,.0f} (target ${taxable["current_tax"].sum():,.0f})')

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Addison TX — 4:1 Split-Rate Tax Impact')

Solving on 3,172 taxable parcels (excluded 100 fully-exempt)
Land millage:        15.2943 per $1,000
Improvement millage: 3.8236 per $1,000
Land/improvement ratio: 4.00:1
Revenue check: $38,335,700 (target $38,335,700)

Addison TX — 4:1 Split-Rate Tax Impact
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential   1388        $321,422        6.9%       $232          $27    3.5%       1.0%            15.4%             2.0%
                   Condominium    583       $-204,961      -19.7%      $-352        $-569  -13.8%     -26.2%            17.5%            58.0%
                    Commercial    488      $1,388,213        7.6%     $2,845       $3,017   24.8%      19.3%            58.4%            19.5%
           Townhome / Rowhouse    427        $-24,916       -2.3%       $-58         $-19   -1.0%      -0.6%             4.0%            11.0%
                   Vacant 

## Validation Summary

| Check | Result |
|---|---|
| Rate | $0.6081/$100 confirmed via FY2025 ACFR |
| Revenue match | Modeled $38.3M vs. certified $34.4M (+11.4%). Residual gap = unmodeled exemptions (over-65 $60K, disabled $60K, and nonprofit commercial not identifiable by owner name). Cannot reduce further without per-parcel exemption flags from DCAD. Revenue-neutral within modeled set. |
| Certification baseline | 2025 Certified EVR (DCAD July 25 2025): real property taxable $5,660M; BPP $960M |
| Parcel filter | Addison GIS portal parcel ID list (3,272 real property parcels). Supersedes PSTLCITY filter (owner mailing address) and centroid-in-polygon (over-captures border parcels). |
| Full exemptions modeled | Town/City of Addison, Dallas ISD, Greenhill School, Whiterock Chapel — identified via owner name matching |
| BPP (excluded by design) | $960.2M taxable, ~$5.8M levy |
| Parcel count | 3,272 real property parcels (100 fully-exempt, 3,172 taxable) |
| Census coverage | 100.0% spatially joined |
| PNGs generated | 7 of 7 |
| SFR median change | +0.5% |
| Vacant land median change | +150.1% |
| Large MFR median change | −26.2% |

## Model Notes

| | |
|---|---|
| Revenue match | Modeled $38.3M vs. certified $34.4M (+11.4%). Gap = unmodeled exemptions (over-65, disabled, unnamed nonprofits). Revenue-neutral within modeled parcel set. |
| Parcel count | 3,272 real property parcels (100 fully-exempt, 3,172 taxable) |
| Parcel source | Addison GIS portal ID list joined to DCAD MapServer (Jan 2025 values) |
| Census coverage | 100.0% spatially joined |
| PNGs generated | 7 of 7 |
| SFR median change | +0.5% |
| Vacant land median change | +150.1% |
| Large MFR median change | −26.2% |

In [6]:
fig, ax = plt.subplots(figsize=(10, 6))
summary = gdf[gdf['full_exmp'] == 0].groupby('PROPERTY_CATEGORY')['tax_change_pct'].median()
summary.sort_values().plot.barh(ax=ax)
ax.set_title('Addison TX — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('Saved category_preview.png')

Saved category_preview.png


## Section 7 — Census Join + Export

In [7]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            # Do NOT merge census_data a second time — census_gdf already carries demographics;
            # a second merge creates median_income_x/y duplicates and zeros demographic output.
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [8]:
from lvt.viz import create_city_report

out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

create_city_report(out_df, CITY_NAME, show=False)
print('Done.')

  ✓ addison: 3,272 rows → ../../analysis/data/addison.csv  [model: split_rate_4to1]


Done.
